In [1]:
# Check whether easydiffraction is installed; install it if needed.
# Required for remote environments such as Google Colab.
import importlib.util

if importlib.util.find_spec('easydiffraction') is None:
    %pip install easydiffraction

# Structure Refinement: PbSO4, XRD

This example demonstrates a more advanced use of the EasyDiffraction
library by explicitly creating and configuring structures and
experiments before adding them to a project. It could be more suitable
for users who are interested in creating custom workflows. This
tutorial provides minimal explanation and is intended for users
already familiar with EasyDiffraction.

The tutorial covers a Rietveld refinement of PbSO4 crystal structure
based on laboratory X-ray powder diffraction data.

## 🛠️ Import Library

In [2]:
from easydiffraction import ExperimentFactory
from easydiffraction import Project
from easydiffraction import StructureFactory
from easydiffraction import download_data

## 🧩 Define Structure

This section shows how to add structures and modify their
parameters.

### Create Structure

In [3]:
struct = StructureFactory.from_scratch(name='pbso4')

### Set Space Group

In [4]:
struct.space_group.name_h_m = 'P n m a'

### Set Unit Cell

In [5]:
struct.cell.length_a = 8.47
struct.cell.length_b = 5.39
struct.cell.length_c = 6.95

### Set Atom Sites

In [6]:
struct.atom_sites.create(
    id='Pb',
    type_symbol='Pb',
    fract_x=0.1876,
    fract_y=0.25,
    fract_z=0.167,
    adp_type='Biso',
    adp_iso=1.37,
)
struct.atom_sites.create(
    id='S',
    type_symbol='S',
    fract_x=0.0654,
    fract_y=0.25,
    fract_z=0.684,
    adp_type='Biso',
    adp_iso=0.3796,
)
struct.atom_sites.create(
    id='O1',
    type_symbol='O',
    fract_x=0.9082,
    fract_y=0.25,
    fract_z=0.5954,
    adp_type='Biso',
    adp_iso=1.9840,
)
struct.atom_sites.create(
    id='O2',
    type_symbol='O',
    fract_x=0.1935,
    fract_y=0.25,
    fract_z=0.5432,
    adp_type='Biso',
    adp_iso=1.4383,
)
struct.atom_sites.create(
    id='O3',
    type_symbol='O',
    fract_x=0.0811,
    fract_y=0.0272,
    fract_z=0.8086,
    adp_type='Biso',
    adp_iso=1.2808,
)

## 🔬 Define Experiments

This section shows how to add experiments, configure their parameters,
and link the structures defined in the previous step.

### Experiment: xrd

#### Download Data

In [7]:
data_path = download_data('meas-pbso4-xray', destination='data')

Getting data...


Data 'meas-pbso4-xray': PbSO4, laboratory X-ray


✅ Data 'meas-pbso4-xray' already present at '../../../data/meas-pbso4-xray.xys'. Keeping existing.


#### Create Experiment

In [8]:
expt = ExperimentFactory.from_data_path(
    name='xrd',
    data_path=data_path,
    radiation_probe='xray',
)

#### Set Instrument

In [9]:
expt.instrument.setup_wavelength = 1.540560
expt.instrument.setup_wavelength_2 = 1.544400
expt.instrument.setup_wavelength_2_to_1_ratio = 0.5

expt.instrument.setup_polarization_coefficient = 0.58
expt.instrument.setup_monochromator_twotheta = 28

expt.instrument.calib_twotheta_offset = -0.0292

#### Set Peak Profile

In [10]:
expt.peak.type = 'pseudo-voigt + berar-baldinozzi asymmetry'

⚠️ Switching peak profile type adds these settings with defaults:                                                                 
   • asym_beba_a0=0.0                                                                                                             
   • asym_beba_a1=0.0                                                                                                             
   • asym_beba_b0=0.0                                                                                                             
   • asym_beba_b1=0.0                                                                                                             


Peak profile type for experiment 'xrd' changed to


pseudo-voigt + berar-baldinozzi asymmetry


In [11]:
expt.peak.broad_gauss_u = 0.0187
expt.peak.broad_gauss_v = -0.0175
expt.peak.broad_gauss_w = 0.0075
expt.peak.broad_lorentz_x = 0
expt.peak.broad_lorentz_y = 0.0655

expt.peak.asym_beba_a0 = -0.2176
expt.peak.asym_beba_b0 = -0.0301

#### Set Excluded Regions

In [12]:
expt.excluded_regions.create(id='1', start=0, end=15)
expt.excluded_regions.create(id='2', start=160, end=180)

#### Set Background

Select background type.

In [13]:
expt.background.type = 'chebyshev'

Background type for experiment 'xrd' changed to


chebyshev


Add Chebyshev background terms.

In [14]:
for id, x, y in [
    ('1', 0, 143.9591),
    ('2', 1, 67.1718),
    ('3', 2, 13.7879),
    ('4', 3, -1.2264),
    ('5', 4, 4.4514),
    ('6', 5, -17.7450),
]:
    expt.background.create(id=id, order=x, coef=y)

#### Set Linked Structures

In [15]:
expt.linked_structures.create(structure_id='pbso4', scale=0.001)

## 📦 Define Project

The project object is used to manage structures, experiments, and
analysis.

### Create Project

In [16]:
project = Project(name='pbso4_xray')

### Add Structure

In [17]:
project.structures.add(struct)

### Add Experiment

In [18]:
project.experiments.add(expt)

## 🚀 Perform Analysis

This section outlines the analysis process, including how to configure
calculation and fitting engines.

### Set Minimizer

In [19]:
project.analysis.minimizer.type = 'bumps (lm)'

Current minimizer changed to


bumps (lm)


### Set Free Parameters

Set structure parameters to be optimized.

In [20]:
struct.cell.length_a.free = True
struct.cell.length_b.free = True
struct.cell.length_c.free = True

# for atom_id in ('Pb', 'S', 'O1', 'O2', 'O3'):
#     atom = struct.atom_sites[atom_id]
#     atom.adp_iso.free = True

Set experiment parameters to be optimized.

In [21]:
expt.linked_structures['pbso4'].scale.free = True

expt.instrument.calib_twotheta_offset.free = True

expt.peak.broad_gauss_u.free = True
expt.peak.broad_gauss_v.free = True
expt.peak.broad_gauss_w.free = True
expt.peak.broad_lorentz_y.free = True

expt.peak.asym_beba_a0.free = True
expt.peak.asym_beba_b0.free = True

for term in expt.background:
    term.coef.free = True

### Run Fitting

In [22]:
project.analysis.fit()

<IPython.core.display.Javascript object>

Standard fitting


📋 Using experiment 🔬 'xrd' for 'single' fitting


🚀 Starting fit process with 'bumps (lm)'...


📈 Goodness-of-fit progress:


,iteration,time (s),χ²,change / status
1,1,0.78,89.42,
2,19,15.76,32.68,63.5% ↓
3,37,31.11,13.59,58.4% ↓
4,55,45.81,4.67,65.6% ↓
5,73,61.09,3.08,34.0% ↓
6,91,76.33,3.07,
7,109,91.39,3.07,
8,131,126.09,3.07,


🏆 Best goodness-of-fit (reduced χ²) is 3.07 at iteration 109


✅ Fitting complete.


In [23]:
project.display.fit.results()

⚙️ Settings used:


,Name,Value,Description
1,max_iterations,1000,Maximum solver iterations.


📋 Least-squares fit results:


,Metric,Value
1,🧪 Minimizer,bumps (lm)
2,✅ Overall status,success
3,⏱️ Fitting time (seconds),126.09
4,📏 Goodness-of-fit (reduced χ²),3.07
5,"📏 R-factor (Rf, %)",6.57
6,"📏 R-factor squared (Rf², %)",6.22
7,"📏 Weighted R-factor (wR, %)",5.28


📈 Refined parameters:


,datablock,category,entry,parameter,units,start,value,s.u.,change
1,pbso4,cell,,length_a,Å,8.4700,8.4807,0.0001,0.13 % ↑
2,pbso4,cell,,length_b,Å,5.3900,5.3991,0.0001,0.17 % ↑
3,pbso4,cell,,length_c,Å,6.9500,6.9604,0.0001,0.15 % ↑
4,xrd,linked_structure,pbso4,scale,,0.0010,0.0010,0.0000,0.02 % ↑
5,xrd,peak,,asym_beba_a0,,-0.2176,-0.2163,0.0054,0.62 % ↓
6,xrd,peak,,asym_beba_b0,,-0.0301,-0.0303,0.0009,0.77 % ↑
7,xrd,peak,,broad_gauss_u,deg²,0.0187,0.0187,0.0008,0.05 % ↓
8,xrd,peak,,broad_gauss_v,deg²,-0.0175,-0.0173,0.0010,1.26 % ↓
9,xrd,peak,,broad_gauss_w,deg²,0.0075,0.0074,0.0003,0.82 % ↓
10,xrd,peak,,broad_lorentz_y,deg,0.0655,0.0650,0.0005,0.69 % ↓


#### Display Correlations

In [24]:
project.display.fit.correlations()

### Display Pattern

In [25]:
project.display.pattern(expt_name='xrd')

In [26]:
project.display.pattern(expt_name='xrd', x_min=77.6, x_max=82.2)

## 💾 Save Project

In [27]:
project.save_as(dir_path='projects/refine-pbso4-xray')

Saving project 📦 'pbso4_xray' to '../../../projects/refine-pbso4-xray'


├── 📄 project.edi


├── 📁 structures/


│   └── 📄 pbso4.edi


├── 📁 experiments/


│   └── 📄 xrd.edi


├── 📁 analysis/


│   └── 📄 analysis.edi


└── 📁 reports/


    └── 📄 pbso4_xray.html
